# Phase 2 Tier 1: Fast XGBoost Importance on Colab

This notebook runs the nonlinear Phase 2 feature-importance workflow on the Tier 1 SPCC feature table.

What it does:
- mounts Google Drive
- loads `spcc_features_tier1.csv`
- trains XGBoost on the fixed `full_tier1_baseline` feature set
- computes test-set permutation importance for `PR-AUC` and `F1`
- records gain importance as the second view
- builds `accept` / `review` / `reject` buckets
- compares `accepted_only`, `accepted_plus_review`, and `full_baseline`
- saves results and the trained model into Google Drive

This version is tuned for speed:
- small XGBoost parameter grid
- early stopping
- 3 permutation repeats
- GPU XGBoost when Colab provides CUDA

In [ ]:
%pip install -q xgboost pandas numpy

In [ ]:
import os
import json
import subprocess
import numpy as np
import pandas as pd
import xgboost as xgb
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

In [ ]:
CSV_PATH = '/content/drive/MyDrive/supernovae_classification/data/processed/spcc_features_tier1.csv'
RESULTS_DIR = '/content/drive/MyDrive/supernovae_classification/results/phase2_tier1'
MODELS_DIR = '/content/drive/MyDrive/supernovae_classification/models/phase2_tier1/xgboost_importance'

RANDOM_STATE = 42
TEST_SPLIT = 0.2
VALIDATION_SPLIT = 0.2
NUM_BOOST_ROUND = 250
EARLY_STOPPING_ROUNDS = 20
PERMUTATION_REPEATS = 3

SURVEY_CONTEXT_FEATURES = [
    'observation_count',
    'time_span',
    'total_snr',
]

PHOTOMETRIC_CORE_FEATURES = [
    'peak_flux_all', 'amplitude_all', 'mean_flux_all', 'std_flux_all',
    'g_peak_flux', 'g_mean_flux', 'g_std_flux', 'g_amplitude',
    'r_peak_flux', 'r_mean_flux', 'r_std_flux', 'r_amplitude',
    'i_peak_flux', 'i_mean_flux', 'i_std_flux', 'i_amplitude',
    'z_peak_flux', 'z_mean_flux', 'z_std_flux', 'z_amplitude',
    'peak_color_g_minus_r', 'peak_color_r_minus_i', 'peak_color_i_minus_z',
]

PEAK_TIME_FEATURES = [
    'time_of_peak_all', 'g_time_of_peak', 'r_time_of_peak', 'i_time_of_peak', 'z_time_of_peak'
]

FEATURE_SETS = {
    'survey_context_only': SURVEY_CONTEXT_FEATURES,
    'photometric_core': PHOTOMETRIC_CORE_FEATURES,
    'full_tier1_baseline': PHOTOMETRIC_CORE_FEATURES + PEAK_TIME_FEATURES + SURVEY_CONTEXT_FEATURES,
}

XGB_PARAM_GRID = [
    {'max_depth': 3, 'eta': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 1.0, 'lambda': 1.0},
    {'max_depth': 4, 'eta': 0.05, 'subsample': 0.9, 'colsample_bytree': 0.9, 'min_child_weight': 1.0, 'lambda': 1.0},
    {'max_depth': 5, 'eta': 0.03, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 2.0, 'lambda': 1.5},
]

assert os.path.exists(CSV_PATH), f'Missing feature table: {CSV_PATH}'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print('CSV_PATH =', CSV_PATH)
print('RESULTS_DIR =', RESULTS_DIR)
print('MODELS_DIR =', MODELS_DIR)
try:
    print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
except Exception as exc:
    print('GPU check failed:', exc)

In [ ]:
df = pd.read_csv(CSV_PATH)
print('Rows:', len(df))
print('Columns:', len(df.columns))
print(df[['label_name']].value_counts().rename('count'))

In [ ]:
def stratified_split_indices(labels, test_size, random_state):
    rng = np.random.default_rng(random_state)
    train_indices = []
    test_indices = []
    for label in np.unique(labels):
        label_indices = np.flatnonzero(labels == label)
        shuffled = label_indices.copy()
        rng.shuffle(shuffled)
        test_count = int(round(len(shuffled) * test_size))
        test_count = min(max(test_count, 1), len(shuffled) - 1)
        test_indices.extend(shuffled[:test_count])
        train_indices.extend(shuffled[test_count:])
    return np.array(sorted(train_indices)), np.array(sorted(test_indices))

def standardize(train_x, other_x):
    mean = train_x.mean(axis=0)
    std = train_x.std(axis=0)
    std[std == 0.0] = 1.0
    return (train_x - mean) / std, (other_x - mean) / std, mean, std

def binary_metrics(y_true, probs, threshold=0.5):
    preds = (probs >= threshold).astype(np.int32)
    y_true = y_true.astype(np.int32)
    tp = int(np.sum((preds == 1) & (y_true == 1)))
    fp = int(np.sum((preds == 1) & (y_true == 0)))
    tn = int(np.sum((preds == 0) & (y_true == 0)))
    fn = int(np.sum((preds == 0) & (y_true == 1)))
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    accuracy = (tp + tn) / len(y_true)
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc_score_numpy(y_true, probs),
        'pr_auc': average_precision_numpy(y_true, probs),
    }

def roc_auc_score_numpy(y_true, scores):
    pos = scores[y_true == 1]
    neg = scores[y_true == 0]
    if len(pos) == 0 or len(neg) == 0:
        return 0.0
    comparisons = (pos[:, None] > neg[None, :]).sum()
    ties = (pos[:, None] == neg[None, :]).sum()
    return float((comparisons + 0.5 * ties) / (len(pos) * len(neg)))

def average_precision_numpy(y_true, scores):
    order = np.argsort(-scores)
    y_sorted = y_true[order]
    tp_cumsum = np.cumsum(y_sorted == 1)
    fp_cumsum = np.cumsum(y_sorted == 0)
    precision = tp_cumsum / np.maximum(tp_cumsum + fp_cumsum, 1)
    positive_total = max(int(np.sum(y_true == 1)), 1)
    recall = tp_cumsum / positive_total
    ap = 0.0
    previous_recall = 0.0
    for p_value, r_value, label in zip(precision, recall, y_sorted):
        if label == 1:
            ap += p_value * (r_value - previous_recall)
            previous_recall = r_value
    return float(ap)

def build_matrix(frame, feature_names):
    x = frame[feature_names].to_numpy(dtype=np.float32)
    y = (frame['label_name'] == 'Ia').astype(np.int32).to_numpy()
    return x, y

labels = (df['label_name'] == 'Ia').astype(np.int32).to_numpy()
trainval_idx, test_idx = stratified_split_indices(labels, TEST_SPLIT, RANDOM_STATE)
train_idx_local, val_idx_local = stratified_split_indices(labels[trainval_idx], VALIDATION_SPLIT, RANDOM_STATE)

train_df = df.iloc[trainval_idx[train_idx_local]].reset_index(drop=True)
val_df = df.iloc[trainval_idx[val_idx_local]].reset_index(drop=True)
trainval_df = df.iloc[trainval_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

print({'train': len(train_df), 'validation': len(val_df), 'test': len(test_df)})

In [ ]:
def base_xgb_params(scale_pos_weight):
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'tree_method': 'hist',
        'verbosity': 0,
        'seed': RANDOM_STATE,
        'scale_pos_weight': scale_pos_weight,
    }
    try:
        params['device'] = 'cuda'
    except Exception:
        pass
    return params

def train_candidate_xgb(train_df, val_df, feature_names, params):
    x_train_raw, y_train = build_matrix(train_df, feature_names)
    x_val_raw, y_val = build_matrix(val_df, feature_names)
    x_train, x_val, mean, std = standardize(x_train_raw, x_val_raw)
    pos_count = float(np.sum(y_train == 1))
    neg_count = float(np.sum(y_train == 0))
    dtrain = xgb.DMatrix(x_train, label=y_train, feature_names=feature_names)
    dval = xgb.DMatrix(x_val, label=y_val, feature_names=feature_names)
    booster = xgb.train(
        params={**base_xgb_params(neg_count / max(pos_count, 1.0)), **params},
        dtrain=dtrain,
        num_boost_round=NUM_BOOST_ROUND,
        evals=[(dval, 'validation')],
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        verbose_eval=False,
    )
    probs = booster.predict(dval, iteration_range=(0, booster.best_iteration + 1))
    return {
        'booster': booster,
        'metrics': binary_metrics(y_val, probs),
        'best_iteration': int(booster.best_iteration + 1),
        'params': params,
    }

def fit_final_xgb(trainval_df, test_df, feature_names, params, num_boost_round):
    x_trainval_raw, y_trainval = build_matrix(trainval_df, feature_names)
    x_test_raw, y_test = build_matrix(test_df, feature_names)
    x_trainval, x_test, mean, std = standardize(x_trainval_raw, x_test_raw)
    pos_count = float(np.sum(y_trainval == 1))
    neg_count = float(np.sum(y_trainval == 0))
    dtrainval = xgb.DMatrix(x_trainval, label=y_trainval, feature_names=feature_names)
    dtest = xgb.DMatrix(x_test, label=y_test, feature_names=feature_names)
    booster = xgb.train(
        params={**base_xgb_params(neg_count / max(pos_count, 1.0)), **params},
        dtrain=dtrainval,
        num_boost_round=num_boost_round,
        verbose_eval=False,
    )
    probs = booster.predict(dtest)
    return booster, binary_metrics(y_test, probs), probs, y_test, x_test, mean, std

def permutation_importance(booster, x_test, y_test, feature_names, repeats=3):
    base_probs = booster.predict(xgb.DMatrix(x_test, feature_names=feature_names))
    base_metrics = binary_metrics(y_test, base_probs)
    rng = np.random.default_rng(RANDOM_STATE)
    rows = []
    for j, feature_name in enumerate(feature_names):
        pr_auc_drops = []
        f1_drops = []
        for _ in range(repeats):
            shuffled = x_test.copy()
            rng.shuffle(shuffled[:, j])
            probs = booster.predict(xgb.DMatrix(shuffled, feature_names=feature_names))
            metrics = binary_metrics(y_test, probs)
            pr_auc_drops.append(base_metrics['pr_auc'] - metrics['pr_auc'])
            f1_drops.append(base_metrics['f1'] - metrics['f1'])
        rows.append({
            'feature': feature_name,
            'mean_pr_auc_drop': float(np.mean(pr_auc_drops)),
            'mean_f1_drop': float(np.mean(f1_drops)),
            'max_pr_auc_drop': float(np.max(pr_auc_drops)),
            'max_f1_drop': float(np.max(f1_drops)),
        })
    rows.sort(key=lambda item: (item['mean_pr_auc_drop'], item['mean_f1_drop']), reverse=True)
    return rows

def gain_importance(booster, feature_names):
    raw_gain = booster.get_score(importance_type='gain')
    raw_weight = booster.get_score(importance_type='weight')
    rows = []
    for feature_name in feature_names:
        rows.append({
            'feature': feature_name,
            'gain': float(raw_gain.get(feature_name, 0.0)),
            'split_count': int(raw_weight.get(feature_name, 0)),
        })
    rows.sort(key=lambda item: item['gain'], reverse=True)
    return rows

def build_feature_buckets(permutation_rows, gain_rows):
    gain_rank = {row['feature']: idx for idx, row in enumerate(gain_rows)}
    buckets = {'accept': [], 'review': [], 'reject': []}
    for row in permutation_rows:
        feature = row['feature']
        gain_position = gain_rank.get(feature, len(gain_rows))
        strong_perm = row['mean_pr_auc_drop'] >= 0.003 or row['mean_f1_drop'] >= 0.003
        weak_perm = row['mean_pr_auc_drop'] <= 0.0005 and row['mean_f1_drop'] <= 0.0005
        strong_gain = gain_position < 10
        weak_gain = gain_position >= 20
        if strong_perm and strong_gain:
            buckets['accept'].append(feature)
        elif weak_perm and weak_gain:
            buckets['reject'].append(feature)
        else:
            buckets['review'].append(feature)
    return buckets

In [ ]:
full_features = FEATURE_SETS['full_tier1_baseline']
candidate_runs = []
for params in XGB_PARAM_GRID:
    run = train_candidate_xgb(train_df, val_df, full_features, params)
    candidate_runs.append(run)

best_run = max(candidate_runs, key=lambda item: item['metrics']['pr_auc'])
print('Selected params:', best_run['params'])
print('Validation metrics:', best_run['metrics'])
print('Best iteration:', best_run['best_iteration'])

final_booster, final_metrics, test_probs, y_test, x_test, mean, std = fit_final_xgb(
    trainval_df,
    test_df,
    full_features,
    best_run['params'],
    best_run['best_iteration'],
)
print('Full baseline test metrics:', final_metrics)

perm_rows = permutation_importance(final_booster, x_test, y_test, full_features, repeats=PERMUTATION_REPEATS)
gain_rows = gain_importance(final_booster, full_features)
buckets = build_feature_buckets(perm_rows, gain_rows)

comparison_sets = {
    'accepted_only': buckets['accept'],
    'accepted_plus_review': buckets['accept'] + buckets['review'],
    'full_baseline': full_features,
}
comparison_results = []
for name, feature_names in comparison_sets.items():
    booster, metrics, _, _, _, _, _ = fit_final_xgb(trainval_df, test_df, feature_names, best_run['params'], best_run['best_iteration'])
    comparison_results.append({
        'name': name,
        'feature_count': len(feature_names),
        'features': feature_names,
        'metrics': metrics,
    })

output = {
    'task': 'phase2_tier1_xgboost_importance',
    'artifact': CSV_PATH,
    'split_manifest': {
        'random_state': RANDOM_STATE,
        'train_count': len(train_df),
        'validation_count': len(val_df),
        'test_count': len(test_df),
    },
    'full_baseline_features': full_features,
    'validation_selection': {
        'params': best_run['params'],
        'best_iteration': best_run['best_iteration'],
        'metrics': best_run['metrics'],
    },
    'full_baseline_test_metrics': final_metrics,
    'importance_scope_note': 'Feature importance is conditional on this preprocessing chain, this XGBoost model family, and this fixed split.',
    'permutation_importance': perm_rows,
    'gain_importance': gain_rows,
    'feature_buckets': buckets,
    'reduced_set_comparisons': comparison_results,
}

result_path = os.path.join(RESULTS_DIR, 'phase2_tier1_xgb_importance.json')
model_path = os.path.join(MODELS_DIR, 'full_tier1_baseline_xgb.json')
final_booster.save_model(model_path)
with open(result_path, 'w') as f:
    json.dump(output, f, indent=2)

print('Saved result_path =', result_path)
print('Saved model_path =', model_path)
print('\nTop permutation importance rows:')
display(pd.DataFrame(perm_rows).head(15))
print('\nTop gain importance rows:')
display(pd.DataFrame(gain_rows).head(15))
print('\nFeature buckets:')
print({k: len(v) for k, v in buckets.items()})
display(pd.DataFrame(comparison_results))

## Notes

- `PR-AUC` is the main ranking target for permutation importance in this notebook.
- Gain importance is provided as a second view, not the only decision source.
- The accept/review/reject buckets here are still model-and-pipeline-conditional, not final astrophysical truth.
- If you want a slower but richer second pass, add SHAP after this fast run is complete.